# Project 3 — RAG Pipeline with LangChain
### LLM Interview Study Assistant | Shruti Agrawal

---

## What are we building?

A RAG (Retrieval Augmented Generation) pipeline that:
1. Reads a document containing LLM Developer interview Q&A
2. Converts the document into searchable vectors
3. When you ask a question — finds the most relevant parts of the document
4. Passes those relevant parts to an LLM to generate an answer
5. Returns an answer grounded in the document — not hallucinated

## Why RAG matters for your interview

RAG is the most commonly deployed LLM application in enterprise settings.
Accenture builds RAG systems for almost every client.
Being able to explain and demonstrate RAG is one of the strongest things
you can show in this interview.

## Before running:
1. Runtime → Change Runtime Type → T4 GPU → Save
2. Upload the file `llm_interview_knowledge.txt` to Colab
   - Click the folder icon on the left sidebar
   - Click the upload button (paper with arrow icon)
   - Select `llm_interview_knowledge.txt`
   - Wait for it to appear in the file list

---

## Cell 1 — Install libraries

New libraries compared to fine-tuning projects:
- **langchain** — the main LangChain library with all building blocks
- **langchain-community** — community integrations including HuggingFace connection
- **faiss-cpu** — the vector database that stores and searches embeddings
- **sentence-transformers** — the model that converts text into vector embeddings
- **pypdf** — reads PDF files (we will use a text file here but good to have)

Everything else you already know from the fine-tuning projects.

In [ ]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers pypdf transformers accelerate bitsandbytes

## Cell 2 — Import everything we need

More imports than before — each one is a building block of the RAG pipeline.
Read each comment carefully.

In [ ]:
# Document loading — reads our text file
from langchain.document_loaders import TextLoader

# Text splitting — cuts the document into smaller chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings — converts text into vectors (numbers representing meaning)
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector store — database that stores and searches our vectors
from langchain_community.vectorstores import FAISS

# Prompt template — reusable prompt with blank spaces
from langchain.prompts import PromptTemplate

# LLM connection — connects to a HuggingFace model
from langchain_community.llms import HuggingFacePipeline

# Chain — connects retriever + prompt + LLM together
from langchain.chains import RetrievalQA

# HuggingFace tools for loading the model
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch

print("All imports successful!")
print("GPU available:", "YES" if torch.cuda.is_available() else "NO — check Runtime settings")

## Cell 3 — Load and split the document

This is Step 1 and 2 of the RAG pipeline:
- Step 1: Load the document from a file
- Step 2: Split it into small chunks

Why split? The LLM has a limited context window — we cannot pass the entire
document to it. We split into chunks and only pass the RELEVANT chunks.
This is the core idea of RAG.

In [ ]:
# ---- STEP 1: LOAD THE DOCUMENT ----

# TextLoader reads a plain text file from the given path
# 'llm_interview_knowledge.txt' is the file we uploaded to Colab
# encoding='utf-8' tells it to read the file using standard text encoding
loader = TextLoader('llm_interview_knowledge.txt', encoding='utf-8')

# .load() actually reads the file and returns a list of Document objects
# Each Document object has .page_content (the text) and .metadata (file info)
documents = loader.load()

# Let's see what we loaded
print(f"Document loaded successfully!")
print(f"Number of document sections: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\nFirst 300 characters of document:")
print("-" * 50)
print(documents[0].page_content[:300])
print("-" * 50)

In [ ]:
# ---- STEP 2: SPLIT INTO CHUNKS ----

# RecursiveCharacterTextSplitter cuts text into pieces
# It tries to split at paragraph breaks first, then sentences, then words
# This is smarter than just cutting every 500 characters blindly
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       # each chunk is maximum 800 characters
                          # 800 is good for Q&A content — fits one full Q&A pair
    chunk_overlap=100,    # 100 characters overlap between consecutive chunks
                          # overlap prevents losing context at chunk boundaries
                          # imagine a sentence starting at end of chunk 1 and
                          # finishing at start of chunk 2 — overlap keeps it intact
    separators=["\n\n", "\n", "Q:", " ", ""]  
                          # try to split at double newlines first (paragraph breaks)
                          # then single newlines, then at Q: markers
                          # then spaces, then anywhere as last resort
)

# Split all loaded documents into chunks
# split_documents takes a list of Document objects and returns a larger list of smaller Document objects
chunks = splitter.split_documents(documents)

print(f"Document split into {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")
print("\nExample chunk (chunk number 3):")
print("-" * 50)
print(chunks[2].page_content)
print("-" * 50)

## Cell 4 — Create embeddings and build vector store

This is Step 3 and 4 of the RAG pipeline:
- Step 3: Convert each chunk into a vector (list of numbers representing meaning)
- Step 4: Store all vectors in a searchable database

After this cell — our entire document lives as vectors in FAISS memory.
We can now search it by MEANING, not just keywords.

This cell will take 2-3 minutes — it processes every chunk.

In [ ]:
# ---- STEP 3: CREATE EMBEDDINGS MODEL ----

# HuggingFaceEmbeddings loads a sentence transformer model
# that converts any text into a vector of numbers
# model_name: 'all-MiniLM-L6-v2' is a small, fast, good quality embedding model
# it converts any text into a vector of 384 numbers
# texts with similar meaning will have similar vectors
# for example: 'leave policy' and 'vacation rules' will have close vectors
# even though they use completely different words
print("Loading embedding model... (takes about 1 minute)")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_kwargs: extra settings passed to the model
    # 'device': 'cuda' means use the GPU for faster embedding generation
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

print("Embedding model loaded!")

# ---- STEP 4: BUILD VECTOR STORE ----

# FAISS.from_documents does two things in one step:
# 1. Takes each chunk and converts it to a vector using our embeddings model
# 2. Stores all those vectors in a FAISS database in memory
# chunks: the list of text pieces we created in Cell 3
# embeddings: the model that converts text to vectors
print("\nConverting chunks to vectors and building search index...")
print("(This processes every chunk — takes 2-3 minutes)")

vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"\nVector store built successfully!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")
print("Each vector has 384 dimensions (numbers)")

## Cell 5 — Test the retriever before adding the LLM

This is an important step most tutorials skip.
Before connecting the LLM, let's test if our retriever is finding the right chunks.
If retrieval is wrong, the LLM answer will be wrong regardless of how good the model is.
Retrieval quality is the most important factor in a RAG system.

In [ ]:
# Convert the vector store into a retriever object
# .as_retriever() makes the vectorstore ready to search
# search_kwargs={"k": 3} means: return the 3 most relevant chunks for each query
# k is how many chunks to retrieve — 3 is a good balance
# too few = might miss relevant info, too many = overwhelms the LLM context
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Test the retriever with a question
# .invoke() runs the retriever — same word as in LangChain chains
# it converts the question to a vector and finds the 3 closest chunk vectors
test_query = "What is LoRA and how does it work?"
retrieved_chunks = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"\nRetrieved {len(retrieved_chunks)} relevant chunks:")
print("=" * 60)

# Loop through each retrieved chunk and print it
# enumerate() gives us both the position number (i) and the chunk (chunk)
# i+1 makes it start from 1 instead of 0 for display
for i, chunk in enumerate(retrieved_chunks):
    print(f"\nChunk {i+1}:")
    print("-" * 40)
    # .page_content is the actual text of the chunk
    # [:400] shows only the first 400 characters so output is not too long
    print(chunk.page_content[:400])
    print("...")

## Cell 6 — Load the LLM

Now we load TinyLlama — same as fine-tuning projects.
But this time we use it differently:
- In fine-tuning: we trained the model
- Here: we use it only for inference (generating answers)

We wrap it in a HuggingFace pipeline, then wrap that in a LangChain object
so it can connect to our RAG chain.

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load in 4-bit quantization to fit in free Colab GPU
# Same BitsAndBytesConfig as fine-tuning projects
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading TinyLlama...")

# Load the model — same as fine-tuning projects
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the tokenizer — same as fine-tuning projects
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Creating HuggingFace pipeline...")

# pipeline() is the simplest way to use a HuggingFace model for inference
# 'text-generation': we want to generate text
# model: the model we loaded above
# tokenizer: the tokenizer we loaded above
# max_new_tokens=300: generate at most 300 new tokens in response
# temperature=0.3: low temperature for focused, factual answers
# do_sample=True: use sampling (not always picking the top token)
# repetition_penalty=1.3: discourages repeating the same words
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.3
)

# HuggingFacePipeline wraps the HuggingFace pipeline into a LangChain compatible object
# This is necessary because LangChain needs its own LLM object format
# Without this wrapper LangChain cannot connect to our model
llm = HuggingFacePipeline(pipeline=hf_pipeline)

print("LLM ready!")

## Cell 7 — Build the RAG chain

This is where everything comes together.
We create a custom prompt template that tells the LLM:
1. You are an interview coach
2. Here are relevant sections from the knowledge document (context)
3. Here is the question
4. Answer based on the context only

Then we connect: retriever → prompt → LLM into one chain.

In [ ]:
# ---- CREATE THE PROMPT TEMPLATE ----

# This template has TWO blanks: {context} and {question}
# {context} will be filled with the retrieved chunks automatically
# {question} will be filled with the user's question automatically
# The instruction 'use only the context provided' reduces hallucination
# because we are telling the model not to make things up

prompt_template = """You are an expert LLM Developer interview coach.
Use only the context provided below to answer the question.
Give a clear, detailed answer. If the answer is not in the context, say "This topic is not covered in the knowledge base."

Context from knowledge base:
{context}

Question: {question}

Answer:"""

# PromptTemplate turns our string into a LangChain prompt object
# input_variables: the list of blanks in our template
# template: the actual prompt string
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

# ---- BUILD THE RAG CHAIN ----

# RetrievalQA.from_chain_type creates the full RAG chain
# It automatically connects: retriever → prompt → llm
# llm: the language model that generates answers
# chain_type="stuff": the simplest method
#   "stuff" means: take all retrieved chunks, stuff them into the prompt, send to LLM
#   other options like "map_reduce" split the work but "stuff" is best for small docs
# retriever: our FAISS retriever that finds relevant chunks
# return_source_documents=True: also return which chunks were used to answer
#   this is useful for transparency — you can see exactly where the answer came from
# chain_type_kwargs: extra settings for the chain
#   here we pass our custom prompt template

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("RAG chain built successfully!")
print("Pipeline: Question → Retriever → Prompt Template → LLM → Answer")

## Cell 8 — Ask interview questions to your RAG assistant!

Now we test the full RAG pipeline.
For each question the system will:
1. Convert question to vector
2. Find 3 most relevant chunks from our knowledge document
3. Pass chunks + question to TinyLlama
4. Return the answer

We also print which source chunks were used — this is called 'grounding'
and it is a key feature of production RAG systems.

In [ ]:
def ask_interview_coach(question):
    """
    Ask the RAG pipeline a question and get a grounded answer.
    
    What happens inside:
    1. question goes to retriever — finds 3 relevant chunks
    2. chunks + question fill the prompt template
    3. filled prompt goes to TinyLlama
    4. TinyLlama generates answer based on the chunks
    5. we return the answer and show the source chunks
    """
    
    print(f"Question: {question}")
    print("-" * 60)
    
    # .invoke() runs the entire RAG chain
    # {"query": question} — RetrievalQA expects the key to be 'query'
    result = rag_chain.invoke({"query": question})
    
    # result is a dictionary with two keys:
    # 'result' — the LLM's generated answer
    # 'source_documents' — the chunks that were retrieved and used
    
    # Extract just the answer part — remove the prompt text if repeated
    answer = result['result']
    
    # Clean up answer — find 'Answer:' and take only what comes after
    if 'Answer:' in answer:
        answer = answer.split('Answer:')[-1].strip()
    
    print(f"Answer: {answer}")
    
    # Show source documents — which parts of the knowledge base were used
    print("\nSource chunks used to answer:")
    # result['source_documents'] is a list of Document objects
    # we loop through them to show where the answer came from
    for i, doc in enumerate(result['source_documents']):
        print(f"  Source {i+1}: {doc.page_content[:150]}...")
    
    print("=" * 60)
    print()


# Test with real interview questions
# These are questions that will actually be asked in your Accenture interview
interview_questions = [
    "What is LoRA and why is it used for fine-tuning?",
    "What is the difference between RAG and fine-tuning?",
    "What is temperature in an LLM?",
]

for q in interview_questions:
    ask_interview_coach(q)


## Cell 9 — Your personal study tool

Now use this as your interview prep tool.
Change the question to anything you want to study or revise.
The RAG pipeline will find the relevant section and explain it.

In [ ]:
# Change this to any interview topic you want to study
your_question = "How do I explain my transition from prompt engineering to LLM development?"

ask_interview_coach(your_question)

---
## What you built — in interview language

*"I built a RAG pipeline using LangChain that reads a domain-specific knowledge document,
splits it into chunks using RecursiveCharacterTextSplitter, converts chunks to vector
embeddings using sentence-transformers, stores them in a FAISS vector store, and retrieves
the top 3 relevant chunks for each query. The retrieved context and question are passed
through a custom PromptTemplate to TinyLlama 1.1B loaded with 4-bit QLoRA quantization.
The system returns grounded answers with source transparency, reducing hallucination
by anchoring responses in verified document content."*

---
## Full RAG pipeline summary — know this diagram:

```
INDEXING (done once):
Document → TextLoader → RecursiveCharacterTextSplitter → chunks
chunks → HuggingFaceEmbeddings → vectors → FAISS vectorstore

QUERYING (done for every question):
Question → Embeddings → Query vector
Query vector → FAISS retriever → Top 3 relevant chunks
Chunks + Question → PromptTemplate → Filled prompt
Filled prompt → TinyLlama → Answer
```

---
## Questions to add to your knowledge document:
Open `llm_interview_knowledge.txt` and add more Q&A pairs.
Re-run Cells 3 and 4 to rebuild the vector store with new content.
Your RAG assistant will immediately know the new information.